<a href="https://colab.research.google.com/github/prishaa09/solarflarephase2/blob/main/Solar_Flare_Phase_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ---------- GOES X-ray data fetch and save ----------
from pathlib import Path
import pandas as pd
from sunpy.net import Fido, attrs as a
from sunpy import timeseries as ts
from google.colab import files

# ---- config ----
START = "2015-01-01 00:00"
END   = "2015-03-31 23:59"
OUT_DIR = Path("/content/data_goes")
CSV_NAME = f"goes_xrs_{START[:10]}_to_{END[:10]}_1min.csv"
CADENCE = "1min"

def fetch_goes_xrs(start, end):
    res = Fido.search(a.Time(start, end), a.Instrument("XRS"))
    files = Fido.fetch(res)
    ts_obj = ts.TimeSeries(files)
    if isinstance(ts_obj, list):
        ts_obj = ts.concatenate(ts_obj)
    return ts_obj

def clean_goes(ts_obj, cadence="1min"):
    df = ts_obj.to_dataframe()
    rename = {}
    for c in df.columns:
        lc = c.lower()
        if ("1-8" in lc) or ("xrsb" in lc) or ("long" in lc): rename[c] = "flux_1_8A"
        if ("0.5-4" in lc) or ("xrsa" in lc) or ("short" in lc): rename[c] = "flux_0.5_4A"
        if ("quality" in lc) or ("qc_flag" in lc): rename[c] = "quality_flag"
        if "sat" in lc: rename[c] = "satellite_id"
    df = df.rename(columns=rename)

    # basic cleanup
    for c in ("flux_0.5_4A", "flux_1_8A"):
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
            df.loc[df[c] < 0, c] = None

    if "quality_flag" in df:
        df = df[df["quality_flag"].isna() | (df["quality_flag"] == 0)]
    df = df.resample(cadence).mean()

    # derived ratio
    if {"flux_1_8A", "flux_0.5_4A"}.issubset(df.columns):
        df["flux_ratio"] = df["flux_0.5_4A"] / df["flux_1_8A"]

    df = df.dropna(subset=["flux_1_8A","flux_0.5_4A"], how="all")
    return df.reset_index().rename(columns={"index":"time_utc"})

def save_csv(df, out_dir, name):
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / name
    df.to_csv(path, index=False)
    print(f"✅ Saved → {path}")
    return path

print(f"Fetching GOES XRS {START} → {END}")
ts_obj = fetch_goes_xrs(START, END)
df = clean_goes(ts_obj, CADENCE)
csv_path = save_csv(df, OUT_DIR, CSV_NAME)

print("Shape:", df.shape)
print(df.head())

# Auto-download the CSV to your computer
files.download(str(csv_path))

Fetching GOES XRS 2015-01-01 00:00 → 2015-03-31 23:59


Files Downloaded:   0%|          | 0/332 [00:00<?, ?file/s]

sci_gxrs-l2-irrad_g13_d20150117_v0-1-0.nc:   0%|          | 0.00/823k [00:00<?, ?B/s]

sci_gxrs-l2-irrad_g13_d20150115_v0-1-0.nc:   0%|          | 0.00/832k [00:00<?, ?B/s]

sci_gxrs-l2-irrad_g13_d20150114_v0-1-0.nc:   0%|          | 0.00/875k [00:00<?, ?B/s]

...

sci_xrsf-l2-avg1m_g15_d20150329_v2-2-1.nc:   0%|          | 0.00/60.2k [00:00<?, ?B/s]

sci_xrsf-l2-avg1m_g15_d20150328_v2-2-1.nc:   0%|          | 0.00/60.0k [00:00<?, ?B/s]

sci_xrsf-l2-avg1m_g15_d20150330_v2-2-1.nc:   0%|          | 0.00/59.9k [00:00<?, ?B/s]

sci_xrsf-l2-avg1m_g15_d20150331_v2-2-1.nc:   0%|          | 0.00/58.0k [00:00<?, ?B/s]

AttributeError: module 'sunpy.timeseries' has no attribute 'concatenate'